# Sequence decoder and scoring helpers

spellF1 matches whole possessions by temporal IoU, so a label sequence that flickers between teams scores badly even
when most frames are right. EVALUATION.md shows one classifier going from 0.05 to 0.24 with smoothing alone.

`HMMSequenceDecoder` works in four steps.

1. A three-state HMM over H, A and D. `label_priors` measures the transition priors on the labelled external matches:
   how long possessions and dead-ball spells last, and how often one team takes the ball straight from the other. The
   notebook prints the public ground truth's priors next to them, for comparison only. The classifier probabilities are the
   emissions, multiplied by `emission_scale` because neighbouring frames are strongly correlated.
2. Viterbi on each half, with frames in a known dead stretch (before the second-half kick-off) fixed to `D`.
3. The decoder merges runs shorter than `min_possession_s` into the neighbour the classifier prefers.
4. An optional `X` rule (`enable_x`): inside a settled spell, at least 2 s from any change, a frame with `max P < 0.40`
   becomes `X`. The submission runs with `enable_x=False`. Frames outside the graded halves are always `X`.

`possession/choose_decoder.ipynb` picks `emission_scale` and `min_possession_s`. `tune_decoder` can still run the grid on
the external validation matches alone.

We depart from the written spec in two places:

* The spec expects few direct `H -> A` changes. The Bundesliga labels disagree: about three quarters of possessions end
  with the other team taking the ball directly (tackles, interceptions). `label_priors` measures that share
  (`handover_share`, 0.77 by default) and `run_modeling.ipynb` stores it with the model.
* For spellF1 the scorer fills an `X` forward with the previous label, so an `X` inside a stable spell changes nothing.
  macroF1, the tie-break, counts it as a miss. We kept the rule as specified behind `enable_x` and left it off.

The notebook also has local versions of both metrics (`spell_f1`, `macro_f1`), written from EVALUATION.md. The scorer
in `evaluation/possession_metrics.py` is the reference; check results with `python evaluation/run_evaluation.py` and
`python evaluation/validate_submission.py`.

In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent   # the repository root: notebooks run from possession/
PIPELINE_DIR = REPO_ROOT / "possession"
ARTIFACT_DIR = PIPELINE_DIR / "artifacts"
OUTPUT_DIR = PIPELINE_DIR / "outputs"     

def load_notebook_defs(filename: str, namespace: dict) -> None:
    loaded = namespace.setdefault("_LOADED_NOTEBOOKS", set())
    if filename in loaded:
        return
    loaded.add(filename)
    nb = json.loads((PIPELINE_DIR / filename).read_text(encoding="utf-8"))
    for i, cell in enumerate(nb["cells"]):
        if cell["cell_type"] == "code" and "defs" in cell.get("metadata", {}).get("tags", []):
            exec(compile("".join(cell["source"]), f"{filename}::cell{i}", "exec"), namespace)

In [ ]:
load_notebook_defs("helper_preprocessing.ipynb", globals())
from scipy.optimize import linear_sum_assignment

In [ ]:
@dataclass
class HMMConfig:
    fps: int = 25
    mean_possession_s: float = 7.3      # measured on the labelled Bundesliga matches (label_priors); run_modeling.ipynb passes its own measurement
    mean_dead_s: float = 12.0
    handover_share: float = 0.77        # share of possession exits that are direct hand-overs (rest -> dead ball); measured on the labelled Bundesliga matches
    dead_to_home: float = 0.5           # after a dead ball, P(home restarts)
    initial: Tuple[float, float, float] = (0.4, 0.4, 0.2)
    emission_scale: float = 0.10        # THE smoothing knob (analogue of the baseline's majority-vote window)
    prob_floor: float = 1e-4
    min_possession_s: float = 2.0
    min_dead_s: float = 2.0
    enable_x: bool = True
    x_conf_threshold: float = 0.40      # spec
    x_guard_s: float = 2.0              # never write X within this distance of a state change (and half edges)
    x_smooth_frames: int = 25           # median-smooth the confidence before thresholding


@dataclass
class DecodeResult:
    labels: np.ndarray        # (T,) '<U1' in {H, A, D, X}
    viterbi: np.ndarray       # (T,) int8 state after merging, before X injection (0 H, 1 A, 2 D, 3 = outside the halves)
    n_x_inside: int


class HMMSequenceDecoder:
    """Viterbi decoding of the frame probabilities with a possession-aware transition matrix + abstention policy."""

    def __init__(self, cfg: Optional[HMMConfig] = None):
        self.cfg = cfg or HMMConfig()
        self.A = self.transition_matrix()
        self.logA = np.log(self.A)
        pi = np.asarray(self.cfg.initial, float)
        self.log_pi = np.log(pi / pi.sum())

    def transition_matrix(self) -> np.ndarray:
        c = self.cfg
        p_exit = 1.0 / (c.mean_possession_s * c.fps)
        q_exit = 1.0 / (c.mean_dead_s * c.fps)
        A = np.zeros((3, 3))
        for s, o in ((0, 1), (1, 0)):                       # H -> {H, A, D}, A -> {A, H, D}
            A[s, s] = 1 - p_exit
            A[s, o] = p_exit * c.handover_share
            A[s, 2] = p_exit * (1 - c.handover_share)
        A[2, 2] = 1 - q_exit
        A[2, 0] = q_exit * c.dead_to_home
        A[2, 1] = q_exit * (1 - c.dead_to_home)
        return A

    def log_emissions(self, probs: np.ndarray) -> np.ndarray:
        c = self.cfg
        p = np.clip(np.nan_to_num(probs, nan=1 / 3), c.prob_floor, 1.0)
        p = p / p.sum(1, keepdims=True)
        return c.emission_scale * np.log(p)

    def viterbi(self, log_e: np.ndarray) -> np.ndarray:
        T = len(log_e)
        delta = np.empty((T, 3))
        psi = np.zeros((T, 3), np.int8)
        delta[0] = self.log_pi + log_e[0]
        cols = np.arange(3)
        for t in range(1, T):
            cand = delta[t - 1][:, None] + self.logA
            psi[t] = cand.argmax(0)
            delta[t] = cand[psi[t], cols] + log_e[t]
        path = np.empty(T, np.int8)
        path[-1] = delta[-1].argmax()
        for t in range(T - 1, 0, -1):
            path[t - 1] = psi[t, path[t]]
        return path

    def merge_short(self, y: np.ndarray, log_e: np.ndarray) -> np.ndarray:
        """Merge every run shorter than the minimum duration into the neighbour with the larger summed log-evidence."""
        c = self.cfg
        min_p, min_d = int(c.min_possession_s * c.fps), int(c.min_dead_s * c.fps)
        y = y.copy()
        for _ in range(100_000):
            starts = np.flatnonzero(np.concatenate([[True], y[1:] != y[:-1]]))
            ends = np.concatenate([starts[1:], [len(y)]])
            lens = ends - starts
            viol = np.flatnonzero(lens < np.where(y[starts] == 2, min_d, min_p))
            if len(viol) == 0 or len(starts) == 1:
                break
            i = viol[np.argmin(lens[viol])]
            a, b = starts[i], ends[i]
            cand = []
            if i > 0:
                cand.append(int(y[starts[i - 1]]))
            if i < len(starts) - 1:
                cand.append(int(y[starts[i + 1]]))
            y[a:b] = max(set(cand), key=lambda s: log_e[a:b, s].sum())
        return y

    def inject_abstention(self, y: np.ndarray, conf: np.ndarray) -> np.ndarray:
        """Codes 0/1/2 plus 3 = X. X only where the classifier is unsure AND we are well inside a possession spell."""
        c = self.cfg
        out = y.astype(np.int8).copy()
        if not c.enable_x:
            return out
        n = len(y)
        guard = int(c.x_guard_s * c.fps)
        bnd = np.zeros(n, bool)
        ch = np.flatnonzero(y[1:] != y[:-1])
        bnd[ch], bnd[ch + 1] = True, True
        dist = ndimage.distance_transform_edt(~bnd) if bnd.any() else np.full(n, np.inf)
        edge = np.minimum(np.arange(n), n - 1 - np.arange(n))
        inside = (dist >= guard) & (edge >= guard) & (y != 2)
        out[inside & (conf < c.x_conf_threshold)] = 3
        return out

    def decode_match(self, probs: np.ndarray, half_blocks, known_dead_ranges=()) -> DecodeResult:
        """probs: (T, 3) = [P(H), P(A), P(D)] for every frame (NaN allowed outside the graded halves)."""
        c = self.cfg
        T = len(probs)
        final = np.full(T, 3, np.int8)
        vit = np.full(T, 3, np.int8)
        for _, a, b in half_blocks:
            p = probs[a:b + 1]
            log_e = self.log_emissions(p)
            for ka, kb in known_dead_ranges:
                lo, hi = max(ka, a) - a, min(kb, b) - a
                if hi >= lo:
                    log_e[lo:hi + 1] = np.array([-30.0, -30.0, 0.0])
            path = self.merge_short(self.viterbi(log_e), log_e)
            maxp = np.where(np.isfinite(p).all(1), np.nan_to_num(p).max(1), 1.0)
            conf = pd.Series(maxp).rolling(c.x_smooth_frames, center=True, min_periods=1).median().to_numpy()
            final[a:b + 1] = self.inject_abstention(path, conf)
            vit[a:b + 1] = path
        labels = np.array(["H", "A", "D", "X"])[final]
        inside = np.isin(np.arange(T), np.concatenate([np.arange(a, b + 1) for _, a, b in half_blocks]))
        return DecodeResult(labels=labels, viterbi=vit, n_x_inside=int(((final == 3) & inside).sum()))


def argmax_labels(probs: np.ndarray, half_blocks) -> np.ndarray:
    """Per-frame argmax with NO temporal model - the flickering reference."""
    out = np.full(len(probs), "X", dtype="<U1")
    for _, a, b in half_blocks:
        out[a:b + 1] = np.array(["H", "A", "D"])[np.nan_to_num(probs[a:b + 1]).argmax(1)]
    return out


In [ ]:
# ---------------------------------------------------------------- local port of the metrics (see EVALUATION.md)
def ffill_labels(p: np.ndarray) -> np.ndarray:
    """Scorer rule: every X takes the last non-X label of the half (back-filled at the head of the half)."""
    isx = p == "X"
    if isx.all():
        return p.copy()
    idx = np.maximum.accumulate(np.where(~isx, np.arange(len(p)), -1))
    idx[idx < 0] = np.flatnonzero(~isx)[0]
    return p[idx]


def label_runs(lbl: np.ndarray) -> List[Tuple[str, int, int]]:
    """Maximal runs of identical labels as (label, start, end_exclusive)."""
    if len(lbl) == 0:
        return []
    starts = np.flatnonzero(np.concatenate([[True], lbl[1:] != lbl[:-1]]))
    ends = np.concatenate([starts[1:], [len(lbl)]])
    return [(str(lbl[s]), int(s), int(e)) for s, e in zip(starts, ends)]


def _possessions(lbl: np.ndarray, offset: int) -> List[Tuple[str, int, int]]:
    return [(l, s + offset, e + offset) for l, s, e in label_runs(lbl) if l in ("H", "A")]


def spell_f1(pred: np.ndarray, gt: np.ndarray, half_blocks, thresholds=(0.3, 0.4, 0.5, 0.6, 0.7)) -> float:
    """F1 of one-to-one possession matching by temporal IoU, averaged over IoU thresholds.
    Assumptions (check against possession_metrics.py): a match needs the same team label; X is forward-filled per half;
    only frames the ground truth covers are scored."""
    P, G = [], []
    for half, a, b in half_blocks:
        g = gt[a:b + 1]
        scored = g != "X"
        if not scored.any():
            continue
        p = ffill_labels(pred[a:b + 1])
        P += _possessions(p[scored], half * 10_000_000)
        G += _possessions(g[scored], half * 10_000_000)
    if not P and not G:
        return 0.0
    if not P or not G:
        return 0.0
    ps, pe = np.array([x[1] for x in P]), np.array([x[2] for x in P])
    gs, ge = np.array([x[1] for x in G]), np.array([x[2] for x in G])
    same = np.array([x[0] for x in P])[:, None] == np.array([x[0] for x in G])[None, :]
    inter = np.clip(np.minimum(pe[:, None], ge[None, :]) - np.maximum(ps[:, None], gs[None, :]), 0, None)
    union = (pe - ps)[:, None] + (ge - gs)[None, :] - inter
    iou = np.where(same, inter / np.maximum(union, 1), 0.0)
    scores = []
    for thr in thresholds:
        ok = (iou >= thr).astype(float)
        r, c = linear_sum_assignment(-ok)
        tp = ok[r, c].sum()
        scores.append(2 * tp / (len(P) + len(G)))
    return float(np.mean(scores))


def macro_f1(pred: np.ndarray, gt: np.ndarray) -> float:
    """Mean per-class F1 over H/A/D on frames the ground truth covers; an X prediction is a miss."""
    m = gt != "X"
    p, g = pred[m], gt[m]
    f1s = []
    for c in "HAD":
        tp = int(((p == c) & (g == c)).sum())
        fp = int(((p == c) & (g != c)).sum())
        fn = int(((p != c) & (g == c)).sum())
        if tp + fp + fn == 0:          # class absent from both truth and prediction
            continue
        f1s.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f1s)) if f1s else 0.0


def score_labels(pred: np.ndarray, gt: np.ndarray, half_blocks) -> Dict[str, float]:
    """spellF1 + macroF1 + possession counts / mean duration (the counts expose flicker immediately)."""
    fps = 25
    n_pred, n_gt, d_pred, d_gt = 0, 0, [], []
    for _, a, b in half_blocks:
        g = gt[a:b + 1]
        sc = g != "X"
        if not sc.any():
            continue
        pp, gg = _possessions(ffill_labels(pred[a:b + 1])[sc], 0), _possessions(g[sc], 0)
        n_pred, n_gt = n_pred + len(pp), n_gt + len(gg)
        d_pred += [(e - s) / fps for _, s, e in pp]
        d_gt += [(e - s) / fps for _, s, e in gg]
    return {"spellF1": round(spell_f1(pred, gt, half_blocks), 4), "macroF1": round(macro_f1(pred, gt), 4),
            "poss_pred": n_pred, "poss_gt": n_gt,
            "mean_dur_pred_s": round(float(np.mean(d_pred)), 1) if d_pred else 0.0,
            "mean_dur_gt_s": round(float(np.mean(d_gt)), 1) if d_gt else 0.0}

In [ ]:
# ---------------------------------------------------------------- priors + tuning from LABELLED matches (the external data)
def ids_to_labels(y: np.ndarray) -> np.ndarray:
    """int labels (0 H, 1 A, 2 D, -1 unknown) -> '<U1' labels with X for unknown."""
    return np.array(["H", "A", "D", "X"])[np.where(y < 0, 3, y)]


def label_priors(label_list, blocks_list, fps: int = 25) -> Dict[str, float]:
    """Possession / dead-ball lengths and how possessions end, measured on labelled sequences (X bridged like the scorer)."""
    poss, dead, exits, handovers = [], [], 0, 0
    for lbl, blocks in zip(label_list, blocks_list):
        for _, a, b in blocks:
            runs = label_runs(ffill_labels(lbl[a:b + 1]))
            for i, (l, s, e) in enumerate(runs):
                if l in ("H", "A"):
                    poss.append((e - s) / fps)
                    if i + 1 < len(runs):
                        exits += 1
                        handovers += runs[i + 1][0] in ("H", "A")
                elif l == "D":
                    dead.append((e - s) / fps)
    poss, dead = np.asarray(poss), np.asarray(dead)
    return {"possessions": int(len(poss)), "mean_possession_s": float(poss.mean()) if len(poss) else np.nan,
            "median_possession_s": float(np.median(poss)) if len(poss) else np.nan,
            "possessions_under_2s_pct": float(100 * (poss < 2).mean()) if len(poss) else np.nan,
            "mean_dead_s": float(dead.mean()) if len(dead) else np.nan,
            "handover_share": float(handovers / max(exits, 1))}


def tune_decoder(prob_list, label_list, blocks_list, base_cfg: HMMConfig,
                 scales=(0.03, 0.05, 0.07, 0.10, 0.15, 0.20, 0.30), min_poss=(1.0, 2.0, 3.0)):
    """Grid-search emission_scale x min_possession_s on LABELLED matches; best mean spellF1 wins (then macroF1, then the longer minimum).
    -> (best HMMConfig, results table sorted best first)."""
    rows = []
    for sc in scales:
        for mp in min_poss:
            cfg = HMMConfig(**{**base_cfg.__dict__, "emission_scale": sc, "min_possession_s": mp, "enable_x": False})
            dec = HMMSequenceDecoder(cfg)
            sp, mf = [], []
            for p, lbl, blocks in zip(prob_list, label_list, blocks_list):
                pred = dec.decode_match(p, blocks).labels
                sp.append(spell_f1(pred, lbl, blocks))
                mf.append(macro_f1(pred, lbl))
            rows.append({"emission_scale": sc, "min_possession_s": mp, "spellF1": round(float(np.mean(sp)), 4),
                         "macroF1": round(float(np.mean(mf)), 4)})
    res = pd.DataFrame(rows).sort_values(["spellF1", "macroF1", "min_possession_s"], ascending=False).reset_index(drop=True)
    best = res.iloc[0]
    cfg = HMMConfig(**{**base_cfg.__dict__, "emission_scale": float(best.emission_scale),
                       "min_possession_s": float(best.min_possession_s)})
    return cfg, res
